# SCHUFA Medallion Pipeline (Lakeflow Spark Declarative Pipeline)

Declarative bronze → silver → gold pipeline consuming the mock SCHUFA AG events that the decoupled "Schufa Event Producer" job drops as JSON files into the landing volume.

* **Bronze** `events_bronze` — streaming table, ingests `/Volumes/test/schufa_mock/landing/events/` via Auto Loader (keeps raw events, including the deliberately malformed ones)
* **Silver** `events_silver` — streaming table: JSON payload flattened and typed, deduplicated on `event_id`, with **data-quality expectations** that drop events without a `person_id` or with an unknown `event_type`
* **Gold** `person_scores` — materialized view: per-person aggregates and a derived mock score (0–100, not a real SCHUFA score)

Publishes to `test.schufa_mock`. Run continuously via the pipeline's continuous schedule.

In [0]:
# Bronze: raw mock events via Auto Loader on the landing volume
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

LANDING_PATH = "/Volumes/test/schufa_mock/landing/events/"

@dp.table(
    name="events_bronze",
    comment="Bronze: raw mock SCHUFA AG events ingested via Auto Loader from the landing volume (incl. malformed events)",
)
def events_bronze():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaHints", "event_ts TIMESTAMP, ingested_at TIMESTAMP, batch_id LONG")
        .load(LANDING_PATH)
    )

In [0]:
# Silver: flattened, typed, deduplicated events — expectations drop invalid records
PAYLOAD_SCHEMA = StructType([
    StructField("person_id", StringType()),
    StructField("person_name", StringType()),
    StructField("city", StringType()),
    StructField("postal_code", StringType()),
    StructField("product_type", StringType()),
    StructField("contract_amount", DoubleType()),
    StructField("payment_status", StringType()),
    StructField("consent_given", BooleanType()),
])

@dp.table(
    name="events_silver",
    comment="Silver: flattened, typed, deduplicated mock events; malformed events dropped by expectations",
)
@dp.expect_or_drop("valid_person_id", "person_id IS NOT NULL")
@dp.expect_or_drop("known_event_type",
                  "event_type IN ('score_request', 'contract_reported', 'payment_event', 'address_change')")
def events_silver():
    return (
        spark.readStream.table("events_bronze")
        .select("event_id", "event_type", "event_ts", "producer", "batch_id", "ingested_at",
                F.from_json("payload", PAYLOAD_SCHEMA).alias("p"))
        .select("event_id", "event_type", "event_ts", "producer", "batch_id", "ingested_at",
                "p.person_id", "p.person_name", "p.product_type",
                F.col("p.contract_amount").cast("decimal(12,2)").alias("contract_amount"),
                "p.payment_status", "p.city", "p.postal_code")
        .withWatermark("event_ts", "1 hour")
        .dropDuplicatesWithinWatermark(["event_id"])
    )

In [0]:
# Gold: per-person aggregates and derived mock score (materialized view)
@dp.materialized_view(
    name="person_scores",
    comment="Gold: per-person aggregates and derived mock score (0-100, not a real SCHUFA score)",
)
def person_scores():
    agg = (
        spark.read.table("events_silver")
        .groupBy("person_id")
        .agg(
            F.last("person_name", ignorenulls=True).alias("person_name"),
            F.last("city", ignorenulls=True).alias("city"),
            F.count(F.when(F.col("event_type") == "score_request", 1)).alias("num_score_requests"),
            F.count(F.when(F.col("event_type") == "contract_reported", 1)).alias("num_contracts"),
            F.coalesce(
                F.sum(F.when(F.col("event_type") == "contract_reported", F.col("contract_amount"))),
                F.lit(0),
            ).cast("decimal(14,2)").alias("total_contract_amount"),
            F.count(F.when(F.col("payment_status") == "late", 1)).alias("num_late_payments"),
            F.count(F.when(F.col("payment_status") == "default", 1)).alias("num_defaults"),
            F.count(F.when(F.col("payment_status") == "on_time", 1)).alias("num_on_time_payments"),
            F.max("event_ts").alias("last_event_at"),
        )
    )
    return agg.select(
        "person_id", "person_name", "city",
        "num_score_requests", "num_contracts", "total_contract_amount",
        "num_late_payments", "num_defaults", "num_on_time_payments", "last_event_at",
        F.least(F.lit(100), F.greatest(F.lit(0),
              F.lit(100)
              - 30 * F.col("num_defaults")
              - 8 * F.col("num_late_payments")
              - F.col("num_score_requests")
              + 2 * F.col("num_on_time_payments"),
        )).cast("int").alias("mock_score"),
    )